# Model: Logistic Regression (TF-IDF)
### IT2011 — Final Evaluation: Model Design, Implementation & Evaluation
**Presented by:** *[Full Name]* — IT25101779
**Model category:** Classification
**Assigned dataset:** Rotten Tomatoes Movie Review Dataset (Cornell)

This notebook covers my individually-owned machine learning model for Progress Review II:
suitability, implementation, hyperparameter tuning across multiple varieties, evaluation, and
a comparison with a conclusion and discussion of limitations.


## Shared Setup

Identical across every member's notebook so each one runs independently. Loads the assigned
dataset and builds the cleaned text column (same cleaning function agreed by the group in
Progress Review I).


In [ ]:
!pip install -q datasets scikit-learn pandas matplotlib seaborn

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, cross_validate, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)

sns.set_style("whitegrid")

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

for df in (train_df, val_df, test_df):
    df["clean_text"] = df["text"].apply(clean_text)

print("Train:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)


## 1. Model Suitability

**Logistic Regression** is a linear classifier that estimates the probability of a review being
positive based on a weighted combination of its TF-IDF word features. It is well suited to this
dataset because: (a) TF-IDF produces high-dimensional, sparse features, which linear models
handle efficiently and without overfitting as readily as more complex models; (b) sentiment in
short movie reviews is often driven by a relatively small set of strongly polarized words (e.g.
"masterpiece", "terrible"), which a linear weighting captures well; and (c) it produces
interpretable coefficients — we can directly inspect which words push a prediction toward
positive or negative.


## 2. Implementation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logreg_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000)),
])
print(logreg_pipeline)


## 3. Hyperparameter Tuning Across Varieties

Three varieties were trained by tuning `max_features` (vocabulary size) and `C` (inverse
regularization strength) via 5-fold GridSearchCV — this satisfies the requirement to assess
performance across different varieties with different tuning.


In [ ]:
param_grid = {
    "tfidf__max_features": [5000, 10000],
    "clf__C": [0.1, 1, 10],
}
grid = GridSearchCV(logreg_pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1, return_train_score=False)
grid.fit(train_df["clean_text"], train_df["label"])

results = pd.DataFrame(grid.cv_results_)[["param_tfidf__max_features", "param_clf__C", "mean_test_score", "std_test_score"]]
results = results.sort_values("mean_test_score", ascending=False)
print("Best parameters:", grid.best_params_)
print("Best cross-validated accuracy:", grid.best_score_)
results


## 4. Evaluation

**Metrics used:** Accuracy and F1 score — appropriate because the dataset is perfectly
class-balanced, so accuracy is meaningful on its own, and F1 confirms neither class is favored.
**Validation method:** 5-fold cross-validation on the training split (above), then a final,
untouched held-out test-set evaluation (below) to confirm the result generalizes.


In [ ]:
best_model = grid.best_estimator_
test_preds = best_model.predict(test_df["clean_text"])

print(classification_report(test_df["label"], test_preds, target_names=["Negative", "Positive"]))

cm = confusion_matrix(test_df["label"], test_preds)
ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(cmap="Blues")
plt.title("Logistic Regression — Test Set Confusion Matrix")
plt.show()


## Overfitting Check: Train vs. Test PerformanceBefore concluding, we check for overfitting by comparing accuracy on the training data itselfagainst the held-out test accuracy already measured above. A large gap between the two(commonly treated as a rule of thumb: more than roughly 5–10 percentage points) suggests themodel has partly memorized patterns specific to the training set rather than learninggeneralizable sentiment signal. This matters here because our training set (8,530 rows) issmall relative to the thousands of TF-IDF feature columns it's matched against — a classicsetup where overfitting risk is elevated.

In [ ]:
train_preds = best_model.predict(train_df["clean_text"])
train_acc = accuracy_score(train_df["label"], train_preds)
test_acc = accuracy_score(test_df["label"], test_preds)
gap = train_acc - test_acc

print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Train-test gap: {gap:.4f}  ({gap*100:.1f} percentage points)")

if gap > 0.10:
    print("Gap exceeds 10 points -- likely overfitting. Try lowering C (more regularization) or reducing tfidf__max_features.")
elif gap > 0.05:
    print("Moderate gap (5-10 points) -- some overfitting; consider a mild increase in regularization.")
else:
    print("Gap is small -- no strong evidence of overfitting.")


## 5. Comparison Across Varieties & Conclusion

The chart below compares cross-validated accuracy across every `(max_features, C)` variety
trained in Section 3.


In [ ]:
plt.figure(figsize=(8, 4))
results_plot = results.copy()
results_plot["config"] = results_plot["param_tfidf__max_features"].astype(str) + " feat, C=" + results_plot["param_clf__C"].astype(str)
sns.barplot(data=results_plot, x="config", y="mean_test_score", color="#3C6E9C")
plt.xticks(rotation=30, ha="right")
plt.ylim(0.7, 0.8)
plt.title("Cross-Validated Accuracy by Hyperparameter Variety")
plt.ylabel("Mean CV accuracy")
plt.xlabel("")
plt.tight_layout()
plt.show()


**Interpretation:** [Fill in after running — describe which variety performed best and by
how much. Typically, C=1 with 10,000 features gives the best balance; very low C (0.1) tends to
under-fit, while the difference between 5,000 and 10,000 features is usually small, suggesting
the extra vocabulary adds limited value once the most informative words are already included.]

## 6. Limitations & Suggested Improvements

- **Bag-of-words blindness:** Logistic Regression on TF-IDF has no sense of word order or
  context, so it can misread sentences with negation or contrastive conjunctions (e.g. "not bad"
  or "great, but...").
- **Suggested improvement:** combine with a contextual model (see IT25101771's DistilBERT
  results) for cases where linear word-weighting alone is insufficient.
